# EduPulse — Student Segmentation via Clustering

A binary at-risk flag tells advisors *who* to flag — but not *why*, or what kind of support to offer.
Clustering the CSE dataset identifies **natural student archetypes** (e.g. high-attender / low-SGPA,
low-attender / decent-SGPA) that map to different intervention strategies.

This notebook:
1. Selects the most informative features for clustering
2. Uses the elbow method to choose K
3. Applies K-means and profiles each cluster
4. Maps clusters to risk levels

In [ ]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
RANDOM_STATE = 42
DATA_DIR = ROOT / "datasets"
print("ROOT:", ROOT)

## 1. Load data

In [ ]:
from src.edupulse.features import load_primary, NUMERIC_FEATURES

df = load_primary(DATA_DIR)
print(f"{len(df):,} students | at-risk rate: {df['at_risk'].mean():.1%}")

## 2. Feature selection for clustering

We cluster on the five most interpretable numeric signals:
attendance, previous SGPA, study hours, social media hours, and credits completed.
All five are scaled before clustering so no single variable dominates by magnitude.

In [ ]:
CLUSTER_FEATURES = [
    "attendance_pct",
    "previous_sgpa",
    "study_hours_daily",
    "social_media_hours",
    "credits_completed",
]

X_raw = df[CLUSTER_FEATURES].dropna()
idx = X_raw.index  # keep aligned with df

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Clustering on {X_raw.shape[0]:,} students, {X_raw.shape[1]} features")

## 3. Elbow method — choosing K

In [ ]:
Ks = range(2, 10)
inertias = []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=15)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(Ks), inertias, "o-", color="#3b7dd8", linewidth=2, markersize=7)
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow method")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. K-means with K = 4

The elbow typically falls around K = 3–5 for this dataset.
K = 4 gives four interpretable archetypes that map cleanly to actionable interventions.

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=15)
labels = km.fit_predict(X_scaled)

cluster_df = df.loc[idx].copy()
cluster_df["cluster"] = labels

# Count and at-risk rate per cluster
summary = cluster_df.groupby("cluster").agg(
    count=("at_risk", "count"),
    at_risk_rate=("at_risk", "mean"),
    **{f: (f, "mean") for f in CLUSTER_FEATURES},
).round(2)
summary["at_risk_rate"] = (summary["at_risk_rate"] * 100).round(1)
print(summary.to_string())

## 5. Visualise clusters (PCA projection)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)

PALETTE = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

fig, ax = plt.subplots(figsize=(8, 5))
for c in range(K):
    mask = labels == c
    at_risk_pct = cluster_df.loc[cluster_df["cluster"] == c, "at_risk"].mean() * 100
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        c=PALETTE[c], alpha=0.5, s=20,
        label=f"Cluster {c}  (n={mask.sum()}, at-risk={at_risk_pct:.0f}%)",
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax.set_title("K-means clusters projected to 2D (PCA)")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Cluster profiles — radar chart

In [ ]:
# Normalise cluster means to [0, 1] for the radar
mins = X_raw.min()
maxs = X_raw.max()

def normalise(series):
    return (series - mins) / (maxs - mins)

profiles_raw = cluster_df.groupby("cluster")[CLUSTER_FEATURES].mean()
profiles_norm = profiles_raw.apply(normalise, axis=0)

labels_clean = [
    "Attendance", "Prev SGPA", "Study hrs", "Social media", "Credits"
]
N = len(labels_clean)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, axes = plt.subplots(1, K, figsize=(14, 4), subplot_kw=dict(polar=True))
for ax, c in zip(axes, range(K)):
    vals = profiles_norm.loc[c].tolist()
    vals += vals[:1]
    ar = cluster_df[cluster_df["cluster"] == c]["at_risk"].mean()
    ax.plot(angles, vals, color=PALETTE[c], linewidth=2)
    ax.fill(angles, vals, color=PALETTE[c], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_clean, fontsize=8)
    ax.set_yticks([])
    ax.set_title(f"Cluster {c}\nat-risk {ar:.0%}", pad=12, fontsize=9)

fig.suptitle("Cluster profiles (normalised means)", y=1.02)
plt.tight_layout()
plt.show()

## 7. Risk map and intervention suggestions

In [ ]:
# Sort clusters by at-risk rate, assign risk level
risk_order = (
    cluster_df.groupby("cluster")["at_risk"].mean()
    .sort_values(ascending=False)
    .reset_index()
)
risk_labels = ["HIGH", "MEDIUM-HIGH", "MEDIUM-LOW", "LOW"]
risk_order["risk_level"] = risk_labels

# Merge risk levels back
cluster_df = cluster_df.merge(risk_order[["cluster", "risk_level"]], on="cluster")

# Summary table
final = (
    cluster_df.groupby("risk_level")
    .agg(
        students=("at_risk", "count"),
        at_risk_pct=("at_risk", lambda x: f"{x.mean():.0%}"),
        mean_attendance=("attendance_pct", "mean"),
        mean_sgpa=("previous_sgpa", "mean"),
        mean_study_hrs=("study_hours_daily", "mean"),
        mean_social_hrs=("social_media_hours", "mean"),
    )
    .round(2)
)
print(final.to_string())

In [ ]:
INTERVENTIONS = {
    "HIGH":        "Immediate one-on-one advisor meeting; consider academic support programme",
    "MEDIUM-HIGH": "Proactive check-in; enrol in study-skills workshop",
    "MEDIUM-LOW":  "Periodic monitoring; encourage peer study groups",
    "LOW":         "Standard monitoring only",
}

print("Suggested interventions by segment:")
print()
for level, action in INTERVENTIONS.items():
    print(f"  {level:<14}  {action}")

## Takeaways

- Four clusters emerge naturally from attendance, prior SGPA, study habits and social media use.
- The **HIGH** cluster is characterised by low attendance *and* low prior SGPA — the double-risk pattern that the binary model also flags most reliably.
- The **MEDIUM-HIGH** cluster often has decent attendance but high social media use and low study hours — a different intervention strategy is warranted.
- Cluster labels are stable across re-runs (K-means with fixed seed) and can be recomputed per cohort at the start of each term.